# final 综合题——时序预测全流程（练习版）

> 数据：`data/load_curve.csv`（90 天 × 24 小时 × 3 台区）。
> 任务：预测 **STATION_A_01** 的小时负荷，七步闭环，全程 assert 兜底。
>
> 注意：挖空块内每条语句保持**单行**；循环/分支写在挖空块之外
> （E4 的 for 已写在挖空块外）。

In [ ]:
from pathlib import Path

import hashlib
import os
import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB",
                                   "Arial Unicode MS", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

DATA = Path("data")

In [ ]:
# 台区选定与指标工具（写在 @@todo 之外）
FEATS = ["hour", "dow", "温度", "是否节假日", "lag24", "lag168", "roll24_mean"]
TARGET = "负荷值"
STATION = "STATION_A_01"


def mae_rmse_r2(yt, yp):
    """回归三件套指标。"""
    return (round(float(mean_absolute_error(yt, yp)), 4),
            round(float(np.sqrt(mean_squared_error(yt, yp))), 4),
            round(float(r2_score(yt, yp)), 4))


def pred_md5(y_pred):
    """预测指纹：落盘验收用。"""
    return hashlib.md5(np.asarray(y_pred).tobytes()).hexdigest()


print("sklearn", sklearn.__version__, "| 目标台区:", STATION)

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def one_line(tag, yt, yp):
    """一行报 MAE / RMSE / R2，返回三元组。"""
    mae, rmse, r2 = mae_rmse_r2(yt, yp)
    print(f"{tag}: MAE={mae} RMSE={rmse} R2={r2}")
    return mae, rmse, r2


print("脚手架就绪：one_line / mae_rmse_r2 / pred_md5")

## 任务 1：清洗

时间戳转换、排序、组内插值，48 个缺口清零。

In [ ]:
raw = pd.read_csv(DATA / "load_curve.csv")
n_missing_before = int(raw["负荷值"].isna().sum())
raw["时间戳"] = pd.to_datetime(raw["时间戳"])
raw = raw.sort_values(["台区编号", "时间戳"]).reset_index(drop=True)
raw["负荷值"] = raw.groupby("台区编号")["负荷值"].transform(lambda s: s.interpolate(limit_direction="both"))
n_missing_after = int(raw["负荷值"].isna().sum())
print("缺失:", n_missing_before, "→", n_missing_after, "| 行数:", len(raw),
      "| 时间范围:", raw["时间戳"].min(), "→", raw["时间戳"].max())

assert (len(raw), raw.shape[1]) == (6480, 5), "90 天 x 24 小时 x 3 台区"
assert n_missing_before == 48 and n_missing_after == 0, "组内插值把 48 个缺口全部补齐"
assert raw["台区编号"].nunique() == 3

## 任务 2：特征工程

hour / dow / lag24 / lag168 / roll24_mean，注意防自泄漏。

In [ ]:
g = raw[raw["台区编号"] == STATION].copy().reset_index(drop=True)
g["hour"] = g["时间戳"].dt.hour
g["dow"] = g["时间戳"].dt.dayofweek
g["lag24"] = g["负荷值"].shift(24)
g["lag168"] = g["负荷值"].shift(168)
g["roll24_mean"] = g["负荷值"].shift(1).rolling(24).mean()
g = g.dropna().reset_index(drop=True)
print("特征工程后:", g.shape)

assert g.shape == (1992, 10), "2160 行 - lag168 甩掉的 168 行 warm-up"
assert g[FEATS + [TARGET]].isna().sum().sum() == 0
assert g["lag24"].iloc[100] == g["负荷值"].iloc[76], "lag24 就是昨天同一时刻"

## 任务 3：时序切分与基线

前 80% 训练；「昨天同时刻」基线必须先立起来。

In [ ]:
cut = int(len(g) * 0.8)
tr, te = g.iloc[:cut], g.iloc[cut:]
Xtr, ytr = tr[FEATS], tr[TARGET]
Xte, yte = te[FEATS], te[TARGET]
base_mae, base_rmse, base_r2 = one_line("基线 lag24", yte, te["lag24"])
print("cut:", cut, "| train:", Xtr.shape, "| test:", Xte.shape)
print("训练段均值:", round(float(ytr.mean()), 4), "| 测试段均值:", round(float(yte.mean()), 4))

assert cut == 1593 and Xtr.shape == (1593, 7) and Xte.shape == (399, 7)
assert base_mae == 48.1487, "『昨天同时刻』这个傻基线并不傻"
assert round(float(ytr.mean()), 4) == 651.1379 and round(float(yte.mean()), 4) == 743.0056, \
    "测试段负荷明显更高：分布漂移本身就是时序任务的常态"

## 任务 4：多模型对比

LR / Ridge / RF / GBR 同台，全部要跑赢基线。

In [ ]:
models = {
    "LR": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RF100": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "GBR100": GradientBoostingRegressor(n_estimators=100, random_state=42),
}
results = {}
for name, model in models.items():
    model.fit(Xtr, ytr)
    results[name] = one_line(name, yte, model.predict(Xte))
print({k: v[0] for k, v in results.items()})

assert results["LR"][0] == 34.2178 and results["Ridge"][0] == 34.2183, \
    "两个线性模型几乎同分：特征已经够『线性友好』"
assert results["RF100"][0] == 31.8473 and results["GBR100"][0] == 30.9282
assert min(v[0] for v in results.values()) < base_mae, \
    "所有模型都跑赢 lag24 基线才算训练有了意义"

## 任务 5：RF 调参

TimeSeriesSplit(3) + 网格 2×3，体会 cv 与 test 的落差。

In [ ]:
gs_rf = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {"n_estimators": [100, 200], "max_depth": [6, 10, None]},
    cv=TimeSeriesSplit(n_splits=3), scoring="neg_mean_absolute_error", n_jobs=-1)
gs_rf.fit(Xtr, ytr)
rf_best_params = gs_rf.best_params_
rf_cv_mae = round(float(-gs_rf.best_score_), 4)
rf_test = one_line("RF 调参后", yte, gs_rf.best_estimator_.predict(Xte))
print("best:", rf_best_params, "| cv MAE:", rf_cv_mae, "| 组合数:", len(gs_rf.cv_results_["params"]))

assert rf_best_params == {"max_depth": 10, "n_estimators": 200}
assert rf_cv_mae == 48.3662, "cv MAE 比 test MAE 惨：训练折更早、更难，是时序 CV 的常态"
assert len(gs_rf.cv_results_["params"]) == 6
assert rf_test[0] == 31.1268 and rf_test[2] == 0.969

## 任务 6：GBR 调参

小步幅 + 多树，冲全场冠军。

In [ ]:
gs_gb = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {"learning_rate": [0.05, 0.1], "n_estimators": [100, 200]},
    cv=TimeSeriesSplit(n_splits=3), scoring="neg_mean_absolute_error", n_jobs=-1)
gs_gb.fit(Xtr, ytr)
gb_best_params = gs_gb.best_params_
gb_cv_mae = round(float(-gs_gb.best_score_), 4)
gb_test = one_line("GBR 调参后", yte, gs_gb.best_estimator_.predict(Xte))
print("best:", gb_best_params, "| cv MAE:", gb_cv_mae)

assert gb_best_params == {"learning_rate": 0.1, "n_estimators": 200}
assert gb_test[0] == 30.0752 and gb_test[2] == 0.9722, "全场冠军：MAE 30.08 / R2 0.9722"
assert gb_test[0] < rf_test[0], "GBR 比 RF 再快 1 个点：时序上残差接力更顺"

## 任务 7：特征重要性

lag24 一家独大意味着什么。

In [ ]:
rf100 = models["RF100"]
imp_rank = sorted(zip(FEATS, rf100.feature_importances_), key=lambda x: -x[1])
top3 = [k for k, _ in imp_rank[:3]]
lag24_imp = round(float(dict(imp_rank)["lag24"]), 4)
print("重要性排序:", [(k, round(float(v), 4)) for k, v in imp_rank])

assert top3 == ["lag24", "lag168", "dow"]
assert lag24_imp == 0.9152, "lag24 独占 91.5%：『明天≈今天』的信息几乎全在昨天同时刻"
assert dict(imp_rank)["温度"] < 0.01, "温度贡献不足 1%——别被『物理直觉』带偏"

## 任务 8：分时段误差解剖

冠军模型在哪些小时翻车。

In [ ]:
best_pred = gs_gb.best_estimator_.predict(Xte)
te_an = te.copy()
te_an["pred"] = best_pred
te_an["err"] = (te_an["pred"] - te_an[TARGET]).abs()
err_by_hour = te_an.groupby("hour")["err"].mean()
worst_hour = int(err_by_hour.idxmax())
best_hour = int(err_by_hour.idxmin())
print("误差最大的小时:", worst_hour, round(float(err_by_hour.max()), 2))
print("误差最小的小时:", best_hour, round(float(err_by_hour.min()), 2))

assert worst_hour == 11 and round(float(err_by_hour.max()), 2) == 51.5, \
    "上午峰 11 点最翻车：负荷尖峰追不上"
assert best_hour == 23 and round(float(err_by_hour.min()), 2) == 13.12, \
    "深夜 23 点最稳：负荷平稳"
assert round(float(err_by_hour.loc[11] - err_by_hour.loc[23]), 2) == 38.37, \
    "峰谷误差差近 4 倍：报告里必须分时段说"

## 任务 9：收官交付

冠军模型落盘 + 指纹验收 + 预测对比图。

In [ ]:
import joblib
best_model = gs_gb.best_estimator_
joblib.dump(best_model, "final_gbr.joblib")
loaded = joblib.load("final_gbr.joblib")
fingerprint_same = pred_md5(best_model.predict(Xte)) == pred_md5(loaded.predict(Xte))
final_mae, final_rmse, final_r2 = mae_rmse_r2(yte, loaded.predict(Xte))
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(te["时间戳"].iloc[:168], yte.iloc[:168], label="真实负荷", lw=1.5)
ax.plot(te["时间戳"].iloc[:168], loaded.predict(Xte)[:168], label="预测负荷", lw=1.5, ls="--")
ax.legend()
ax.set_title("STATION_A_01 未来 7 天负荷预测（GBR）")
ax.set_ylabel("kW")
fig.tight_layout()
fig.savefig("final_ts_preview.png", dpi=120)
plt.close(fig)
os.remove("final_ts_preview.png")
print("模型大小:", os.path.getsize("final_gbr.joblib"), "bytes | 指纹一致:", fingerprint_same)
print(f"最终成绩: MAE={final_mae} RMSE={final_rmse} R2={final_r2}")

assert os.path.exists("final_gbr.joblib")
assert fingerprint_same, "落盘模型指纹一致才算交付"
assert (final_mae, final_rmse, final_r2) == (30.0752, 39.8829, 0.9722)
os.remove("final_gbr.joblib")  # 校验完即清

## 完成标准

所有 assert 通过 = 综合题满分骨架跑通。自查：滚动窗口为什么先 shift(1)、
时序切分为什么不能用 train_test_split、cv 分数比 test 惨是不是调参失败、
基线在报告里的作用是什么。